# Chroma Vector Database
- Chroma는 대규모 언어 모델(LLM) 애플리케이션 구축을 위해 설계된 AI 네이티브 **오픈 소스 벡터 데이터베이스**다.    
- 임베딩 저장소, 쿼리 및 검색 등의 핵심 기능을 제공하여 개발자들이 효율적으로 작업할 수 있도록 돕는다. 
- https://www.trychroma.com/
  
## Chroma의 주요 특징

- **오픈 소스 라이선스** 
  - Apache 2.0 라이선스에 따라 제공되어 누구나 자유롭게 사용하고 수정할 수 있다. 
- **다양한 개발 환경 지원**
  -  Python 및 JavaScript/TypeScript SDK를 지원하여 다양한 Langchain 과 연동하여 활용할 수 있다. 
- **유연한 데이터 저장 옵션**
  -  HTTP 방식, 디스크 저장 방식, 인메모리 방식을 선택하여 데이터를 저장할 수 있어 사용자 입장에서 매우 편리하다. 
- **간편한 사용법** 
  - 설치 및 사용법이 매우 간단하여 빠르게 프로토타입을 개발하고 검증할 수 있다. 

## 설치
- <del>pip로 chromadb 설치시 **windows**에서는 c컴파일러 관련되어 에러가 난다. **conda 를 이용해 설치한다.**</del>
- `conda install conda-forge::chromadb`
- `pip install chromadb`
- `pip install langchain-chroma`

# Chroma API 를 이용해 연동
- https://docs.trychroma.com/

In [1]:
import chromadb

In [2]:
from uuid import uuid4

# 추가할 데이터
document_list = [
        "This is a document about pineapple",
        "This is a document about oranges",
        "This is a document about sports",
        "This is a document about langchain",
]
ids = [str(uuid4()) for _ in range(len(document_list))]
# 디비에 저장할 때 지정할 각 문서들의 ID 생성.
ids

['973cf0ea-c587-4df7-9636-73cea3d97e7d',
 'c724164b-d58d-4c97-b56e-27b670a226b4',
 '13b86e87-836d-4c69-b5ee-7c1673c7eb7e',
 'e94fad3c-f695-48c0-995e-c41356f9a903']

In [7]:
#  외부 Embedding 모델 
from dotenv import load_dotenv
import chromadb.utils.embedding_functions as embedding_functions
import os

print(load_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
                api_key=OPENAI_API_KEY,
                model_name="text-embedding-3-small"
            )


True


In [8]:
# collection - Database
# Chroma DB와 연결
import chromadb
client = chromadb.Client()   # InMemory DB (데이터를 메모리에 저장)
# client = chromadb.PersistentClient(path="vector_store/chroma/my_db") # Local 파일에 저장.
# client = chromadb.HttpClient(host="ip주소", port=8000) # 서버로 서비스하는 chromadb에 연결

# Collection(Database)을 생성
collection_name = "test_db"
collection = client.create_collection(
    name=collection_name,
    get_or_create=True, # collection이 있으면 연결, 없으면 생성. 
                        #  (False: 이미 있는 collection이면 Exception)
    metadata={"hnsw:space":"cosine"}, # 코사인 유사도록 계산.
    embedding_function=openai_ef
)


In [9]:
######### 데이터 추가
collection.add(documents=document_list, ids=ids)

In [10]:
##### 유사도 검색
result = collection.query(
    query_texts=["deeplearning", "hawaii"],  #질문
    n_results=2,                 # 검색 결과 수
)


In [11]:
result

{'ids': [['ac9c134b-c468-40a8-8c77-dac8ca093b19',
   '555d3bf2-c0cd-4562-b735-326b92a12871'],
  ['555d3bf2-c0cd-4562-b735-326b92a12871',
   'ac9c134b-c468-40a8-8c77-dac8ca093b19']],
 'embeddings': None,
 'documents': [['This is a document about langchain',
   'This is a document about pineapple'],
  ['This is a document about pineapple',
   'This is a document about langchain']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None], [None, None]],
 'distances': [[0.7781278491020203, 0.8145824670791626],
  [0.7556753754615784, 0.8807869553565979]]}

# Langchain을 이용해 Chroma 연동

## Data 준비

In [10]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)
document_list = [document_1, document_2, document_3, document_4, document_5, 
                document_6, document_7, document_8, document_9, document_10,  ]
ids = [str(uuid4()) for _ in range(len(document_list))]

In [4]:
%pip install langchain-chroma

  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.72.1 requires protobuf<7.0.0,>=6.30.0, but you have protobuf 5.29.5 which is incompatible.


## Vector Store 생성, 연결
- Chroma.from_documents()
  - VectorStore를 초기화(생성)하고 문서를 추가한다.
  - persist_directory를 지정하지 않으면 메모리에 저장된다.
- Chroma()
  - VectorStore와 연결.

In [14]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

COLLECTION_NAME = "example2"  # 컬렉션 이름 (RDB의 Database개념)
PERSISTENT_PATH = 'vector_store/chroma/example_db' # 저장할 로컬 경로

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. 연결(생성)하면서 document들을 저장(upsert)
vector_store = Chroma.from_documents(
    documents=document_list,
    ids=ids,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSISTENT_PATH
)

In [15]:
# 2. 연결(생성)
vector_store2 = Chroma(
    embedding_function=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSISTENT_PATH
)

## VectorStore 정보 확인

In [16]:
coll = vector_store._collection # 연결된 collection 정보를 확인
coll

Collection(name=example2)

In [17]:
coll.count() # 저장된 데이터개수

23

## Add (추가)

In [18]:
document_11 = Document(
    page_content="랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.",
    metadata={"source": "tweet"},
    # id=10,
)

document_12 = Document(
    page_content="랭체인은 체인 구조를 사용하여 여러 LLM 작업을 연결하고, 이를 통해 더 복잡하고 맞춤화된 자연어 처리 애플리케이션을 개발할 수 있게 합니다",
    metadata={"source": "tweet"},
    # id=10,
)

document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게!",
    metadata={"source": "news"},
    # id=10,
)

In [19]:
vector_store.add_documents(
    [document_11, document_12, document_13], 
    # ids=[str(uuid4()), str(uuid4()), str(uuid4())]
)

['ba96eaa3-480f-4acf-b9d8-d09a4705ca53',
 '3b2cbb69-51a6-41b8-b447-8e3f21612ab9',
 'f6a49668-641d-4445-8fa0-729764dce927']

In [20]:
coll.count()

26

## Update(갱신)

In [21]:
new_document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게 처리할 수있는 Framework.",
    metadata={"source": "news"},
    id=10,
)

In [22]:
vector_store.update_document(
    document_id="0fc85d7f-1642-4c5b-9b67-0dbb9fbc2f0e", # 바꿀 문서의 ID
    document=new_document_13  # 바꿀 내용을 가진 Document객체
)

In [23]:
update_document_12 = Document(
    page_content="랭체인은 체인 구조를 사용하여 여러 LLM 작업을 연결할 수있다.",
    metadata={"source": "website"} # tweet -> website
)

update_document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게!",
    metadata={"source": "news"}
)

update_docs = [update_document_12, update_document_13]
update_ids = ['8e66ec2a-5224-46c1-866e-4f7a027c174f','0fc85d7f-1642-4c5b-9b67-0dbb9fbc2f0e']

vector_store.update_documents(documents=update_docs, ids=update_ids)
# 한번에 여러개 update

In [24]:
# coll.get()  # 전체 저장된 문서를 조회
vector_store.get()

{'ids': ['853ad991-3f8c-4ec4-96f1-3b2bed7d0e8c',
  '64d1d964-2cc5-469a-b0fc-2a6d57c34c35',
  'dc090a17-e66f-41af-a287-6cc6307f6451',
  '66715546-55cb-432c-ad60-6467463d885a',
  '48c6bebf-7448-4fda-91d9-62b458a7da16',
  '3d30fdad-47a7-4d6d-915e-4b9964e77e18',
  '03785328-ccb0-47e6-a7b6-473a69daf688',
  '2930e6f8-fdca-49e2-9652-946a65bba6fe',
  'ed6df82a-11db-475d-94b7-8812271fd788',
  '546eb9a1-5111-4f4c-bb95-5eb975913113',
  'eb956ac8-3d1c-47f7-8684-f6122eeee321',
  'e3207f1b-f2aa-4603-be83-cfc37586a85d',
  '1b9f7616-c6b1-4077-ab31-bcb3500303dd',
  'fbfdf5f2-84f1-4dc0-9e08-893b320184d6',
  '0661b012-6d97-401a-9968-3be18e58c0c5',
  '31015d9d-0a1a-49fb-94c5-5ab048b5368d',
  'b19edf01-32e1-459a-805c-38a3f3896814',
  'c6990e6f-2562-458b-b6a2-437cc9b967ab',
  'efcf06e1-06e4-4b56-b244-52fc609b9294',
  'f9d04bca-aed1-4d02-a981-86030e2b8e73',
  '7d988788-6ffb-4558-839b-f4be041a57e2',
  '6b5727c5-82da-4856-9713-9a916fdd269c',
  '980e3d93-15b8-4e8e-93e4-3cafad5c28e0',
  'ba96eaa3-480f-4acf-b9d8-

## Delete(삭제)

In [25]:
del_ids = ['209677ba-ebbd-4a55-b41d-35b03b755866','f56bf32e-ae8a-49dc-a40e-fcb0eb794adf']
vector_store.delete(ids=del_ids)   # [삭제할 문서들의 id들]

In [26]:
coll.count()

26

## Query(조회)
- `similarity_search(query, k, filter)`
  - 저장되 있는 item들 중 질의와 가장 유사한 것 k개를 찾는다. 
  - 찾은 결과를 filter 조건으로 필터링 한다. filter 조건은 meta-data의 정보를 이용한다.
  - 질의어(query)는 text(자연어)로 입력한다.
- `similarity_search_with_score(query, k, filter)`
  - 저장되 있는 item들 중 질의와 가장 유사한 것 k개를 찾아 유사도 점수와 함께 반환
- `similarity_search_by_vector(embedding, k, filter)`
  - Embedding Vector 를 질의로 입력한다. (질의(query)를 문장이 아니라 embedding vector로 입력.) 

In [27]:
results = vector_store.similarity_search(
    query="Langchain이란 무엇인가요?",
    k=3, # 조회개수
)
results

[Document(id='dc090a17-e66f-41af-a287-6cc6307f6451', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='31015d9d-0a1a-49fb-94c5-5ab048b5368d', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='ba96eaa3-480f-4acf-b9d8-d09a4705ca53', metadata={'source': 'tweet'}, page_content='랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.')]

In [28]:
results = vector_store.similarity_search_with_score(
    query="아침에 무엇을 먹으면 좋을까?",
    k=3, # 조회개수
    # filter={"source":"tweet"}  # metadata의 source키값이 tweet(source==tweet)
    filter={"source":{"$ne":"news"}}   # source가 news가 아닌 것들.
    #{metadata key: {"연산자":"값"}}
    # {"age":{"$gt", 30}}  # age > 30
)
# 1. filter에 설정과 metadata를 비교해서 조회
# 2. 1에서 조회된 문서들과 query간의 유사도를 체크
results

[(Document(id='853ad991-3f8c-4ec4-96f1-3b2bed7d0e8c', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  1.2472906112670898),
 (Document(id='fbfdf5f2-84f1-4dc0-9e08-893b320184d6', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  1.2473175525665283),
 (Document(id='3d30fdad-47a7-4d6d-915e-4b9964e77e18', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
  1.7841017246246338)]

In [44]:
# self-shot
from langchain_chroma import Chroma
COLLECTION_NAME="vector_store1"				# 각 벡터 DB의 이름
PERSIST_DIRECTORY="./vector_store/chroma"	# 저장할 디렉토리 위치

vector_store = Chroma.from_documents(
	documents=document_list,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)


In [43]:
vector_store._collection.count()
vector_store._collection.get('5')

{'ids': ['5'],
 'embeddings': None,
 'documents': ["Wow! That was an amazing movie. I can't wait to see it again."],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'tweet'}]}

In [46]:
vector_store.search("영화 추천", search_type="similarity")

[Document(id='5', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='6', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(id='8', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]